## setup

In [ ]:
# import stuff
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [ ]:
# load data from folder
folder = Path('../data/processed')

# filtered datasets with mortgage rates
listings = pd.read_csv(folder / 'wk4_5_listings_clean.csv', low_memory = False)

In [ ]:
listings.head()
listings.shape

In [ ]:
date_cols = ['PurchaseContractDate',
             'ListingContractDate',
             'ContractStatusChangeDate']

listings[date_cols] = listings[date_cols].apply(pd.to_datetime, errors = 'coerce')

## feature engineering

In [ ]:
# ft eng
listings['price_ratio'] = listings['ClosePrice'] / listings['OriginalListPrice']

# normalizes price across sizes
listings['price_per_sqft'] = listings['ClosePrice'] / listings['LivingArea']

# captures full price reduction history
listings['close_to_original_list_ratio'] = listings['ClosePrice'] / listings['OriginalListPrice']

# measures time from listing to accepted offer
listings['listing_to_contract_days'] = listings['PurchaseContractDate'] - listings['ListingContractDate']

# check that engineered columns were created
listings[['price_ratio',
          'price_per_sqft',
          'close_to_original_list_ratio',
          'listing_to_contract_days',
          ]].head()

In [ ]:
listings[['price_ratio',
          'price_per_sqft',
          'close_to_original_list_ratio',
          'listing_to_contract_days',
          ]].isna().sum()

## add school districts

In [ ]:
# add school districts
school_gdf = gpd.read_file('../data/school_districts.geojson')

In [ ]:
# filter by unified school district
filtered_school_gdf = school_gdf[school_gdf['DistrictType'] == 'Unified']

# only get districtname col & geometry for merging
filtered_school_gdf = filtered_school_gdf[['DistrictName', 'geometry']]

# check that changes have been made
filtered_school_gdf.head()

In [ ]:
listings_gdf = gpd.GeoDataFrame(
    listings,
    geometry = gpd.points_from_xy(listings['Longitude'], listings['Latitude']),
    crs = 'EPSG:4326'
)

listings_gdf.head()

In [ ]:
# standardize coordinate systems so they match
# otherwise you get a crs mismatch error
if listings_gdf.crs != school_gdf.crs:
    listings_gdf = listings_gdf.to_crs(school_gdf.crs)

In [ ]:
# spatially merge datasets
merged = gpd.sjoin(
    listings_gdf,
    filtered_school_gdf,
    how = 'left',
    predicate = 'within'
)

merged[['Latitude', 'Longitude', 'DistrictName']].head()

In [ ]:
district_null_pct = merged['DistrictName'].isna().sum() / merged['DistrictName'].shape[0]

print('Percentage of missing values in DistrictName column:',
      round(district_null_pct, 4)
      )

## segment analysis

In [ ]:
metrics = ['PropertySubType',
            'CountyOrParish',
            'MLSAreaMajor',
            'ListOfficeName',
            'BuyerOfficeName']

print('LISTINGS DATASET:')

for metric in metrics:
    print(f'Summary statistics for {metric}:')
    print(listings[metric].describe(), '\n')

In [ ]:
merged.columns

In [ ]:
school_cols = ['ElementarySchool', 'MiddleOrJuniorSchool', 'HighSchool', 'HighSchoolDistrict', 'DistrictName']

print(merged[school_cols].isna().sum(), '\n')
merged[school_cols].head()

''' 
ElementarySchool        534503
MiddleOrJuniorSchool    534388
HighSchool              511835
HighSchoolDistrict      181981
DistrictName            201106
dtype: int64 
'''

In [ ]:
merged.shape